In [1]:
import pandas as pd
import numpy as np

In [ ]:
# Choose fallback policy for teams missing in the previous season:
#   - "worst"  : use the worst rank from (league, prev_season)
#   - "middle" : use floor(#teams / 2) from (league, prev_season)
fallback_policy = "middle"   # or "middle"

END_PATH  = "../csv/end_of_season/end_of_season_us.csv"   # league,season,team_id,...,rank
GAMES_PATH = "../csv/us_combined_data.csv"  # season,date,league,team1,team2,result,score1,score2
end = pd.read_csv(END_PATH)
games = pd.read_csv(GAMES_PATH)

In [3]:
# We will simulate each game in season S using ranks from previous season S-1
# We will starting from the second earliest season of each league
games["prev_season"] = games["season"] - 1

league_min_season = (
    end.groupby("league", as_index=False)["season"]
       .min()
       .rename(columns={"season": "league_min_season"})
)
games = games.merge(league_min_season, on="league", how="left")
games_sim = games[
    games["prev_season"].notna() & (games["prev_season"] >= games["league_min_season"])
].copy()

In [ ]:
# PREVIOUS-SEASON STANDINGS
end_prev = end[["league", "season", "team_id", "rank"]].rename(
    columns={"season": "prev_season"}
)

# Per (league, prev_season) stats for fallback rank
season_team_stats = (
    end_prev.groupby(["league", "prev_season"])
    .agg(num_teams=("team_id", "nunique"), worst_rank=("rank", "max"))
    .reset_index()
)

if fallback_policy == "worst":
    season_team_stats["fallback_rank"] = season_team_stats["worst_rank"].astype(int)
elif fallback_policy == "middle":
    season_team_stats["fallback_rank"] = np.floor(season_team_stats["num_teams"] / 2).astype(int)
else:
    raise ValueError("fallback_policy must be 'worst' or 'middle'")

# Attach the chosen fallback_rank to each game
games_sim = games_sim.merge(
    season_team_stats[["league", "prev_season", "fallback_rank"]],
    on=["league", "prev_season"],
    how="left"
)

In [ ]:
games_sim = games_sim.merge(
    end_prev.rename(columns={"team_id": "team1", "rank": "rank1"}),
    on=["league", "prev_season", "team1"],
    how="left"
)
games_sim = games_sim.merge(
    end_prev.rename(columns={"team_id": "team2", "rank": "rank2"}),
    on=["league", "prev_season", "team2"],
    how="left"
)

games_sim["rank1"] = games_sim["rank1"].fillna(games_sim["fallback_rank"]).astype(int)
games_sim["rank2"] = games_sim["rank2"].fillna(games_sim["fallback_rank"]).astype(int)

# DETERMINE RESULT & SCORES
# - result = 1 if team1 has better (lower) rank than team2
# - result = 0 if same rank
# - result = -1 if team2 has better (lower) rank
# - score1/score2 are 1/0 or 0/1 accordingly; ties -> 0/0
games_sim["result"] = np.where(
    games_sim["rank1"] < games_sim["rank2"], 1,
    np.where(games_sim["rank1"] > games_sim["rank2"], -1, 0)
)

games_sim["score1"] = (games_sim["result"] == 1).astype(int)
games_sim["score2"] = (games_sim["result"] == -1).astype(int)

result_df = games_sim[[
    "season", "date", "league", "team1", "team2", "result", "score1", "score2"
]].copy()

In [ ]:
result_df.to_csv("../csv/us_combined_pure_skill_middle.csv", index=False)